In [1]:
%pip install scikit-learn
%pip install accelerate>=1.1.0

import numpy as np
import evaluate
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)



Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip

[notice] A new release of pip is available: 25.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


In [2]:

dataset = load_dataset("tweet_eval", "emotion")

label_names = ["anger", "joy", "optimism", "sadness"]
id2label = {i: label for i, label in enumerate(label_names)}
label2id = {label: i for i, label in enumerate(label_names)}


In [3]:
checkpoint = "distilbert/distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

def tokenize_function(batch):
    return tokenizer(batch["text"], truncation=True)

tokenized_dataset = dataset.map(tokenize_function, batched=True)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

accuracy = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return accuracy.compute(predictions=predictions, references=labels)

model = AutoModelForSequenceClassification.from_pretrained(
    checkpoint,
    num_labels=4,
    id2label=id2label,
    label2id=label2id
)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert/distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [4]:
#training
training_args = TrainingArguments(
    output_dir="distilbert_tweeteval_emotion",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    logging_steps=50,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics
)


trainer.train()




Epoch,Training Loss,Validation Loss,Accuracy
1,0.723059,0.666077,0.751337


Epoch,Training Loss,Validation Loss,Accuracy
1,0.723059,0.666077,0.751337
2,0.472734,0.616123,0.772727


Epoch,Training Loss,Validation Loss,Accuracy
1,0.723059,0.666077,0.751337
2,0.472734,0.616123,0.772727
3,0.325798,0.607903,0.778075


Epoch,Training Loss,Validation Loss,Accuracy
1,0.723059,0.666077,0.751337
2,0.472734,0.616123,0.772727
3,0.325798,0.607903,0.778075


TrainOutput(global_step=612, training_loss=0.5967921143263774, metrics={'train_runtime': 35.7175, 'train_samples_per_second': 273.563, 'train_steps_per_second': 17.134, 'total_flos': 105960880963560.0, 'train_loss': 0.5967921143263774, 'epoch': 3.0})

In [5]:
test_results = trainer.evaluate(tokenized_dataset["test"])
distilbert_accuracy = test_results["eval_accuracy"]
print("Compatre DistilBert vs LTMs:")
print("DistilBERT Accuracy:", distilbert_accuracy)
print("LTM Accuracy from Task 3: 0.655")
difference = distilbert_accuracy - 0.655
print("Difference:", difference)
print("\n\n\n")

predictions = trainer.predict(tokenized_dataset["test"])
pred_labels = np.argmax(predictions.predictions, axis=-1)

for i in range(5):
    print("Tweet:", dataset["test"][i]["text"])
    print("True:", id2label[dataset["test"][i]["label"]])
    print("Pred:", id2label[pred_labels[i]])
    print()

Training Loss,Validation Loss,Epoch,Accuracy
0.325798,0.558003,3,0.805067


Compatre DistilBert vs LTMs:
DistilBERT Accuracy: 0.8050668543279381
LTM Accuracy from Task 3: 0.655
Difference: 0.1500668543279381






Tweet: #Deppression is real. Partners w/ #depressed people truly dont understand the depth in which they affect us. Add in #anxiety &amp;makes it worse
True: sadness
Pred: sadness

Tweet: @user Interesting choice of words... Are you confirming that governments fund #terrorism? Bit of an open door, but still...
True: anger
Pred: anger

Tweet: My visit to hospital for care triggered #trauma from accident 20+yrs ago and image of my dead brother in it. Feeling symptoms of #depression
True: sadness
Pred: sadness

Tweet: @user Welcome to #MPSVT! We are delighted to have you! #grateful #MPSVT #relationships
True: joy
Pred: joy

Tweet: What makes you feel #joyful?
True: joy
Pred: joy

